# Notebook 08: Cold Start and Exploration

## Why This Matters

Netflix adds 500+ new titles per year. Spotify has 100,000 new tracks uploaded every day. TikTok has millions of new creators posting content daily. Every one of these items starts with zero interaction history — it's invisible to a collaborative filtering system. This is the cold start problem, and it's arguably the most important practical challenge in production recommender systems.

Beyond cold start, recommender systems face a fundamental exploration-exploitation tension: always showing the most popular items maximizes short-term engagement but creates filter bubbles, stagnates new item discovery, and makes the model's data increasingly self-reinforcing. Understanding how to balance exploration and exploitation — drawing on decades of bandit algorithm research — is the difference between a system that learns and one that ossifies.

## Background

### Cold Start Variants

**New user cold start**: A user signs up with no history. Options:
- Onboarding questionnaire: ask for top genres, artists, or explicit preferences
- Demographic-based priors: if available and ethical
- Popular items: safe, boring, but at least not harmful
- Content-based warm start: use contextual signals (device, time, referrer)

**New item cold start**: A new item enters the catalog with no interactions. Options:
- Content-based warm start: embed item's metadata/text/image into the embedding space via an auxiliary encoder
- Injection into exploration slots: guarantee some impressions to new items
- Staff picks / editorial curation: high-quality items get a manual boost

**New platform cold start**: No users, no items, no data. Import a social graph, license a dataset, or start with content-based filtering until enough data accumulates.

### Content-Based Warm Start for New Items

The idea: train an auxiliary model that maps item content features (text, image, metadata) to a point in the same embedding space as the collaborative filtering item embeddings. When a new item enters, encode its content to get a "warm" initial embedding. As users interact with it, gradually shift toward the learned collaborative embedding.

In practice: train a content encoder alongside the CF model, using the CF item embeddings as targets. The content encoder learns: "what position in the embedding space does this item's content correspond to?"

### Exploration vs. Exploitation: The Bandit Problem

A recommendation slot is a sequential decision problem:
- **Greedy (pure exploitation)**: Always show the item with highest predicted CTR. Fast convergence to popular items, no learning about uncertain items.
- **Epsilon-greedy**: With probability $\varepsilon$, show a random item; with probability $1-\varepsilon$, show the best known item. Simple but explores uniformly — wastes budget on clearly bad items.
- **Thompson Sampling**: Maintain a distribution over expected reward for each item. Sample from each item's distribution and show the item with the highest *sample*. Naturally explores uncertain items more.
- **UCB (Upper Confidence Bound)**: Show the item that maximizes $\hat{\mu}_i + c\sqrt{\log t / N_i}$ where $N_i$ is the number of times item $i$ has been shown. "Optimistic in the face of uncertainty."

Thompson Sampling and UCB have $O(\log T)$ regret — optimal for the multi-armed bandit problem. Epsilon-greedy has $O(\sqrt{T})$ regret — asymptotically worse.

### Diversity: Maximal Marginal Relevance (MMR)

Even a perfect ranker will produce boring, repetitive slates if you just take the top-K items by score. MMR (Carbonell & Goldstein, 1998) reranks by balancing:
1. Relevance to the user query
2. Diversity from already-selected items

$$\text{MMR} = \arg\max_{i \in R \setminus S} [\lambda \cdot \text{sim}(i, q) - (1-\lambda) \max_{j \in S} \text{sim}(i, j)]$$

where $S$ is the already-selected set, $R$ is the candidate set, and $q$ is the query. $\lambda$ controls the relevance-diversity tradeoff.

In [ ]:
%pip install -q numpy pandas matplotlib scipy scikit-learn tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import beta
from sklearn.metrics.pairwise import cosine_similarity
import warnings, os, requests, zipfile, io, time
from collections import defaultdict

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
movies = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1] + list(range(5,24)),
                     names=["item_id","title"] + [f"genre_{i}" for i in range(19)])

genre_names = ["Unknown","Action","Adventure","Animation","Children","Comedy",
               "Crime","Documentary","Drama","Fantasy","Film-Noir","Horror",
               "Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"]

n_users = ratings.user_id.max()
n_items = ratings.item_id.max()
print(f"Users: {n_users}, Items: {n_items}")
print("Ready.")

## 1. Cold Start Problem: Interaction Coverage Analysis

Let's quantify the cold start challenge: what fraction of items get enough interactions to be reliably represented by collaborative filtering?

In [ ]:
# Interaction coverage: items with < N ratings are cold start
ratings_per_item = ratings.groupby("item_id").size()
thresholds = [1, 5, 10, 20, 50, 100]

print("Cold start analysis by interaction threshold:")
print(f"{'Threshold':<12} {'Cold Items':<12} {'% of Catalog':<14} {'% of Interactions':<20}")
print("-" * 60)
for t in thresholds:
    cold_items = (ratings_per_item < t).sum()
    cold_frac = cold_items / n_items
    cold_interactions = ratings_per_item[ratings_per_item < t].sum()
    cold_int_frac = cold_interactions / len(ratings)
    print(f"< {t:4d} ratings: {cold_items:7,} items  ({cold_frac:.1%} of catalog, "
          f"{cold_int_frac:.2%} of interactions)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sorted_pop = ratings_per_item.sort_values(ascending=False).values
axes[0].loglog(range(1, len(sorted_pop)+1), sorted_pop, color="steelblue")
axes[0].axhline(10, color="red", linestyle="--", label="Cold start threshold (10)")
axes[0].fill_between(range(1, len(sorted_pop)+1), sorted_pop, 10,
                      where=(sorted_pop < 10), alpha=0.3, color="red", label="Cold start items")
axes[0].set_xlabel("Item rank (log scale)")
axes[0].set_ylabel("Number of interactions (log scale)")
axes[0].set_title("Item Popularity: Log-Log Scale")
axes[0].legend()

# Cumulative coverage
cumulative = np.cumsum(sorted_pop) / np.sum(sorted_pop)
axes[1].plot(np.arange(1, len(cumulative)+1) / n_items * 100, cumulative * 100,
             color="coral", linewidth=2)
axes[1].axvline(20, color="gray", linestyle="--", alpha=0.7, label="Top 20% of items")
y_20 = cumulative[int(0.2 * n_items) - 1] * 100
axes[1].axhline(y_20, color="gray", linestyle=":", alpha=0.7)
axes[1].text(22, y_20-5, f"{y_20:.0f}% of interactions", fontsize=9, color="gray")
axes[1].set_xlabel("% of item catalog (by popularity)")
axes[1].set_ylabel("% of interactions")
axes[1].set_title("Lorenz Curve: Interaction Concentration")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/cold_start_analysis.png", dpi=120)
plt.show()

## 2. Content-Based Warm Start for New Items

The warm start approach: represent items by their content features (genre one-hot vectors in this case) and project them into the item embedding space learned by collaborative filtering.

In production with text/image items: train a content encoder (e.g., sentence-transformers for text, CLIP for images) and map its output to the CF embedding space via a learned linear projection.

In [ ]:
# Build a simple content-based item representation using genre features
genre_cols = [f"genre_{i}" for i in range(19)]
item_content = movies[["item_id"] + genre_cols].copy().set_index("item_id")
item_content = item_content.fillna(0)

# Compute item-item content similarity (genre overlap)
content_matrix = item_content.values.astype(np.float32)

# Normalize for cosine similarity
norms = np.linalg.norm(content_matrix, axis=1, keepdims=True)
content_normalized = content_matrix / np.clip(norms, 1e-8, None)

# Simulate CF item embeddings (in practice these come from a trained MF model)
# For demo: compute SVD-based embeddings from the ratings matrix
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

n_u = n_users
n_i = n_items
R = np.zeros((n_u, n_i), dtype=np.float32)
for _, row in ratings.iterrows():
    R[int(row.user_id)-1, int(row.item_id)-1] = row.rating
R_sparse = csr_matrix(R)

k = 50
U, sigma, Vt = svds(R_sparse, k=k)
cf_item_embeddings = (Vt.T * sigma).astype(np.float32)  # (n_items, k)

print(f"CF item embeddings: {cf_item_embeddings.shape}")
print(f"Content features:   {content_normalized.shape}")

# Train a content→CF mapping: linear regression
# For each item with enough interactions, map content features → CF embedding
min_interactions = 10
popular_items = ratings_per_item[ratings_per_item >= min_interactions].index.tolist()
cold_items_list = ratings_per_item[ratings_per_item < min_interactions].index.tolist()

print(f"\nItems with ≥{min_interactions} interactions (training data): {len(popular_items)}")
print(f"Cold start items (< {min_interactions} interactions):          {len(cold_items_list)}")

# Fit linear mapping: content features → CF embedding
from sklearn.linear_model import Ridge

train_item_ids = [i for i in popular_items if i <= n_items and i <= len(item_content)]
X_content_train = content_normalized[np.array(train_item_ids) - 1]  # (N, 19)
y_cf_train = cf_item_embeddings[np.array(train_item_ids) - 1]       # (N, 50)

content_to_cf = Ridge(alpha=1.0)
content_to_cf.fit(X_content_train, y_cf_train)

# Predict CF embeddings for cold start items
test_cold_ids = [i for i in cold_items_list[:100] if i <= n_items and i <= len(item_content)]
X_content_cold = content_normalized[np.array(test_cold_ids) - 1]
cold_embeddings = content_to_cf.predict(X_content_cold)

print(f"\nWarm-start embeddings computed for {len(test_cold_ids)} cold items")
print(f"Embedding shape: {cold_embeddings.shape}")

# Validate: for popular items, check if content→CF prediction is close to true CF
from sklearn.metrics.pairwise import cosine_similarity as csim
true_embs = cf_item_embeddings[np.array(train_item_ids[:50]) - 1]
pred_embs = content_to_cf.predict(X_content_train[:50])
cos_sims = [csim(true_embs[i:i+1], pred_embs[i:i+1])[0,0] for i in range(50)]
print(f"Content→CF mapping quality (cosine sim, popular items): {np.mean(cos_sims):.3f} ± {np.std(cos_sims):.3f}")

## 3. Multi-Armed Bandit: Thompson Sampling

We simulate a recommendation bandit problem. We have a set of items with unknown true CTRs. The goal is to maximize total clicks over many rounds while also discovering which items are actually best.

Thompson Sampling maintains a Beta distribution per item (Beta(α, β) where α=successes+1, β=failures+1) and samples from each distribution to make decisions. The Beta distribution naturally narrows as more data is collected — exploration decreases as confidence increases.

In [ ]:
class BanditEnvironment:
    """Simulated recommendation bandit with known true CTRs."""

    def __init__(self, n_items=20, seed=42):
        rng = np.random.default_rng(seed)
        # True CTRs: mix of popular (high CTR) and niche (low CTR) items
        self.true_ctrs = np.concatenate([
            rng.beta(5, 2, size=3),     # 3 genuinely good items
            rng.beta(2, 5, size=7),     # 7 mediocre items
            rng.beta(1, 10, size=10),   # 10 poor items (new/unknown)
        ])
        self.n_items = n_items
        self.optimal_item = np.argmax(self.true_ctrs)
        self.optimal_ctr = self.true_ctrs[self.optimal_item]

    def interact(self, item_id):
        """Returns 1 (click) or 0 (no click) based on true CTR."""
        return int(np.random.random() < self.true_ctrs[item_id])


class ThompsonSampling:
    def __init__(self, n_items):
        self.alpha = np.ones(n_items)  # successes + 1
        self.beta_ = np.ones(n_items)  # failures + 1

    def select(self):
        samples = np.random.beta(self.alpha, self.beta_)
        return np.argmax(samples)

    def update(self, item_id, reward):
        self.alpha[item_id] += reward
        self.beta_[item_id] += 1 - reward


class UCBBandit:
    def __init__(self, n_items):
        self.counts = np.zeros(n_items)
        self.values = np.zeros(n_items)
        self.t = 0

    def select(self):
        self.t += 1
        # Ensure every item is tried at least once
        untried = np.where(self.counts == 0)[0]
        if len(untried) > 0:
            return untried[0]
        ucb = self.values + np.sqrt(2 * np.log(self.t) / self.counts)
        return np.argmax(ucb)

    def update(self, item_id, reward):
        self.counts[item_id] += 1
        n = self.counts[item_id]
        self.values[item_id] = ((n-1) * self.values[item_id] + reward) / n


class EpsilonGreedy:
    def __init__(self, n_items, epsilon=0.1):
        self.epsilon = epsilon
        self.counts = np.zeros(n_items)
        self.values = np.zeros(n_items)

    def select(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(len(self.values))
        return np.argmax(self.values)

    def update(self, item_id, reward):
        self.counts[item_id] += 1
        n = self.counts[item_id]
        self.values[item_id] = ((n-1) * self.values[item_id] + reward) / n


def run_simulation(strategy, env, n_rounds=2000):
    """Run bandit simulation, return cumulative regret and click rate."""
    cumulative_regret = []
    total_regret = 0
    click_rates = []
    total_clicks = 0

    for t in range(n_rounds):
        item = strategy.select()
        reward = env.interact(item)
        strategy.update(item, reward)
        total_regret += env.optimal_ctr - env.true_ctrs[item]
        total_clicks += reward
        cumulative_regret.append(total_regret)
        click_rates.append(total_clicks / (t + 1))

    return cumulative_regret, click_rates


# Run all strategies on same environment
env = BanditEnvironment(n_items=20, seed=42)
print(f"Environment: {env.n_items} items, optimal item={env.optimal_item}, optimal CTR={env.optimal_ctr:.3f}")
print(f"True CTRs: {np.round(env.true_ctrs, 3)}")

N_ROUNDS = 3000
strategies = {
    "Thompson Sampling": ThompsonSampling(env.n_items),
    "UCB": UCBBandit(env.n_items),
    "ε-greedy (ε=0.1)": EpsilonGreedy(env.n_items, epsilon=0.1),
    "ε-greedy (ε=0.3)": EpsilonGreedy(env.n_items, epsilon=0.3),
}

results = {}
for name, strategy in strategies.items():
    regret, click_rate = run_simulation(strategy, env, n_rounds=N_ROUNDS)
    results[name] = {"regret": regret, "click_rate": click_rate}
    print(f"{name:<25}: Final regret={regret[-1]:.1f}, Avg CTR={click_rate[-1]:.3f}")

## 4. Bandit Strategy Comparison

Visualizing cumulative regret and click rate over time reveals the exploration-exploitation tradeoff in action. Thompson Sampling's Bayesian posterior naturally decays exploration — near optimal performance with minimal parameter tuning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple"]

for (name, res), color in zip(results.items(), colors):
    axes[0].plot(res["regret"], label=name, color=color, linewidth=2)
    axes[1].plot(res["click_rate"], label=name, color=color, linewidth=2)

axes[0].axhline(0, color="black", linestyle="--", alpha=0.3)
axes[0].set_xlabel("Round")
axes[0].set_ylabel("Cumulative Regret")
axes[0].set_title("Cumulative Regret: Exploration vs Exploitation\n(Lower is better)")
axes[0].legend(fontsize=9)

axes[1].axhline(env.optimal_ctr, color="black", linestyle="--", alpha=0.5, label="Optimal CTR")
axes[1].set_xlabel("Round")
axes[1].set_ylabel("Running Average CTR")
axes[1].set_title("Average Click-Through Rate Over Time\n(Higher is better)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/bandit_comparison.png", dpi=120)
plt.show()

# Thompson Sampling posterior evolution
ts = ThompsonSampling(env.n_items)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, n_rounds in zip(axes, [50, 500, 3000]):
    ts_temp = ThompsonSampling(env.n_items)
    env_temp = BanditEnvironment(n_items=20, seed=42)
    for _ in range(n_rounds):
        item = ts_temp.select()
        ts_temp.update(item, env_temp.interact(item))

    x = np.linspace(0, 1, 200)
    for i in range(min(5, env.n_items)):
        from scipy.stats import beta as beta_dist
        y = beta_dist.pdf(x, ts_temp.alpha[i], ts_temp.beta_[i])
        ax.plot(x, y, label=f"Item {i} (CTR={env.true_ctrs[i]:.2f})", alpha=0.8)
    ax.set_title(f"Beta Posteriors after {n_rounds} rounds")
    ax.set_xlabel("Expected CTR")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)

plt.suptitle("Thompson Sampling: Beta Posterior Narrowing Over Time")
plt.tight_layout()
plt.savefig("/tmp/thompson_posteriors.png", dpi=120)
plt.show()

## 5. Diversity: Maximal Marginal Relevance (MMR)

A pure relevance ranker will often return very similar items — if you love action movies, you'll get the top-10 most popular action movies, but they might all be similar. MMR reranks candidates by trading off relevance against diversity.

In [ ]:
def mmr_rerank(item_ids, relevance_scores, item_embeddings, lam=0.5, K=10):
    """
    Maximal Marginal Relevance reranking.

    Args:
        item_ids: candidate item indices
        relevance_scores: predicted relevance for each candidate
        item_embeddings: embedding matrix (n_items, d)
        lam: lambda tradeoff (1.0 = pure relevance, 0.0 = pure diversity)
        K: number of items to select

    Returns:
        ordered list of selected item indices
    """
    candidates = list(zip(item_ids, relevance_scores))
    selected = []
    remaining = candidates.copy()

    for _ in range(min(K, len(candidates))):
        if not selected:
            # First item: highest relevance
            best = max(remaining, key=lambda x: x[1])
        else:
            # MMR: balance relevance and diversity
            selected_embs = item_embeddings[[s[0] for s in selected]]  # (|S|, d)
            best_score = -np.inf
            best = None

            for item_id, rel in remaining:
                item_emb = item_embeddings[item_id:item_id+1]  # (1, d)
                # Max similarity to already-selected items
                if len(selected_embs) > 0:
                    sims = cosine_similarity(item_emb, selected_embs)[0]
                    max_sim = sims.max()
                else:
                    max_sim = 0

                mmr_score = lam * rel - (1 - lam) * max_sim
                if mmr_score > best_score:
                    best_score = mmr_score
                    best = (item_id, rel)

        selected.append(best)
        remaining = [x for x in remaining if x[0] != best[0]]

    return [s[0] for s in selected]


# Demonstrate: Genre diversity with vs without MMR
# Simulate a user who likes action movies
item_pop = ratings.groupby("item_id")["rating"].mean()
genre_cols = [f"genre_{i}" for i in range(19)]
movies_with_genres = movies.set_index("item_id")[genre_cols]

# Items with genre data
valid_items = [i for i in range(1, n_items+1)
               if i in movies_with_genres.index and i in item_pop.index]

# Fake relevance scores: action movies score highest
action_col = genre_cols[1]  # Action genre
relevance_sim = {}
for item_id in valid_items[:200]:
    is_action = movies_with_genres.loc[item_id, action_col]
    base_score = item_pop.get(item_id, 2.5)
    relevance_sim[item_id] = base_score * (1.5 if is_action else 0.8)

candidate_ids = sorted(valid_items[:200], key=lambda x: relevance_sim.get(x, 0), reverse=True)[:50]
rel_scores = [relevance_sim.get(i, 0) for i in candidate_ids]

# Build item embeddings from content
content_embs_all = content_normalized  # (n_items, 19) genre features

# Pure top-K
top_k_pure = candidate_ids[:10]

# MMR reranked
top_k_mmr = mmr_rerank(
    [i-1 for i in candidate_ids],  # 0-indexed
    rel_scores,
    content_embs_all,
    lam=0.5, K=10
)
top_k_mmr = [i+1 for i in top_k_mmr]  # back to 1-indexed

# Count genre diversity
def genre_count(item_list, genre_col):
    return sum(movies_with_genres.loc[i, genre_col] for i in item_list if i in movies_with_genres.index)

print("\nDiversity comparison: pure relevance vs MMR (λ=0.5)")
print(f"{'Genre':<12} {'Pure Top-10':>12} {'MMR Top-10':>12}")
print("-" * 38)
for g_name, g_col in zip(genre_names[1:8], genre_cols[1:8]):
    pure_count = genre_count(top_k_pure, g_col)
    mmr_count = genre_count(top_k_mmr, g_col)
    print(f"{g_name:<12} {pure_count:>12} {mmr_count:>12}")

print("\nPure top-10 items:")
for i in top_k_pure[:5]:
    row = movies[movies.item_id == i]
    if len(row):
        print(f"  {row.iloc[0].title[:40]}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| New item cold start | Items with zero interactions are invisible to CF — need content warm start |
| Warm start | Map content features → CF embedding space via auxiliary model |
| Exploration-exploitation | Always greedy → filter bubble; explore → better learning |
| Epsilon-greedy | Simple but wastes budget on clearly bad items; $O(\sqrt{T})$ regret |
| Thompson Sampling | Beta posterior per item; naturally decays exploration; $O(\log T)$ regret |
| UCB | Optimistic in uncertainty: $\hat{\mu}_i + c\sqrt{\log t / N_i}$; principled |
| Filter bubble | Over-exploiting popular items creates a self-reinforcing loop |
| MMR | Maximal Marginal Relevance — rerank for diversity while preserving relevance |
| $\lambda$ in MMR | 1.0 = pure relevance, 0.0 = pure diversity, 0.5 = balanced |

### Common Pitfalls
- No new item injection strategy — new items get zero impressions, no learning, immediate cold start death
- Using exploration for all users equally — high-value users should get less exploration
- Setting $\varepsilon$ too high in epsilon-greedy — too much random exploration hurts engagement metrics
- MMR with $\lambda$ too low — diversity can overpower relevance and hurt CTR
- Ignoring the warm start quality — poor content features produce uninformative initial embeddings